# Simplified Two-Lens System Analysis

This notebook demonstrates:
1. Forward model using 2 lenses with ray tracing
2. Computing ABCD transfer matrix via differentiation
3. Collins FFT diffraction model
4. Comparison and Bayesian optimization

## System Setup
- Input: Circular aperture (1 μm diameter) with padding
- Output: 10mm × 10mm detector, 256×256 pixels
- Method: Compare ray-traced vs diffraction-computed fields

In [ ]:
import sys
sys.path.insert(0, '../../src')

import jax
import jax.numpy as jnp
import jax.scipy as jsp
import numpy as np
import matplotlib.pyplot as plt

from temgym_core.components import Lens, Detector
from temgym_core.ray import Ray
from temgym_core.propagator import FreeSpaceParaxial
from temgym_core.run import solve_model
from temgym_core.constants import energy2wavelength

jax.config.update("jax_enable_x64", True)

print("Imports successful")

## Constants and Parameters

In [ ]:
# System parameters
VOLTAGE = 300e3  # 300 kV
WAVELENGTH = energy2wavelength(VOLTAGE)  # in meters

# Aperture parameters
APERTURE_RADIUS = 0.5e-6  # 0.5 μm radius (1 μm diameter)
INPUT_SIZE = 5e-6  # 5 μm total input grid (with padding)
INPUT_PIXELS = 512  # Higher resolution for input

# Output detector parameters
OUTPUT_SIZE = 10e-3  # 10 mm × 10 mm
OUTPUT_PIXELS = 256  # 256 × 256

# Lens parameters (example - will optimize later)
Z1 = 0.0  # First lens position
Z2 = 0.1  # Second lens position (100 mm)
Z3 = 0.5  # Detector position (500 mm)
F1 = 0.05  # First lens focal length (50 mm)
F2 = 0.15  # Second lens focal length (150 mm)

print(f"Wavelength: {WAVELENGTH*1e12:.4f} pm")
print(f"Aperture diameter: {2*APERTURE_RADIUS*1e6:.2f} μm")
print(f"Input grid: {INPUT_SIZE*1e6:.2f} μm × {INPUT_PIXELS} pixels")
print(f"Output grid: {OUTPUT_SIZE*1e3:.2f} mm × {OUTPUT_PIXELS} pixels")

## 1. Forward Model with Ray Tracing

Build a two-lens system and trace rays through it.

In [ ]:
def build_two_lens_model(z1, z2, z3, f1, f2):
    """Build a two-lens optical system.
    
    Parameters
    ----------
    z1, z2, z3 : float
        Axial positions of lens1, lens2, and detector
    f1, f2 : float
        Focal lengths of lens1 and lens2
        
    Returns
    -------
    model : list
        List of components [lens1, lens2, detector]
    """
    lens1 = Lens(z=z1, focal_length=f1)
    lens2 = Lens(z=z2, focal_length=f2)
    detector = Detector(
        z=z3,
        pixel_size=(OUTPUT_SIZE/OUTPUT_PIXELS, OUTPUT_SIZE/OUTPUT_PIXELS),
        shape=(OUTPUT_PIXELS, OUTPUT_PIXELS),
        centre=(0.0, 0.0)
    )
    return [lens1, lens2, detector]

# Build the model
model = build_two_lens_model(Z1, Z2, Z3, F1, F2)
print(f"Model built with {len(model)} components")
print(f"  Lens 1: z={Z1}m, f={F1}m")
print(f"  Lens 2: z={Z2}m, f={F2}m")
print(f"  Detector: z={Z3}m")

## 2. Compute ABCD Transfer Matrix

Use JAX automatic differentiation to compute the 5×5 ABCD matrix.
We differentiate through the forward model using a single input ray.

In [ ]:
def get_abcd_matrix(z1, z2, z3, f1, f2):
    """Compute the ABCD transfer matrix for the two-lens system.
    
    Parameters
    ----------
    z1, z2, z3 : float
        Axial positions
    f1, f2 : float
        Focal lengths
        
    Returns
    -------
    abcd : jnp.ndarray, shape (5, 5)
        ABCD transfer matrix from input to detector
    """
    # Create a ray at origin
    ray = Ray.origin()
    
    # Build model
    model = build_two_lens_model(z1, z2, z3, f1, f2)
    
    # Get all ABCD matrices through the system
    abcd_matrices = solve_model(ray, model)
    
    # The cumulative ABCD matrix is the product of all matrices
    # For our purposes, we want the total transfer from input to output
    cumulative_abcd = abcd_matrices[0]
    for i in range(1, len(abcd_matrices)):
        cumulative_abcd = abcd_matrices[i] @ cumulative_abcd
    
    return cumulative_abcd

# Compute ABCD matrix
abcd = get_abcd_matrix(Z1, Z2, Z3, F1, F2)
print("ABCD Transfer Matrix:")
print(abcd)
print(f"\nKey parameters:")
print(f"  A (magnification x): {abcd[0, 0]:.6f}")
print(f"  A (magnification y): {abcd[1, 1]:.6f}")
print(f"  B (defocus parameter x): {abcd[0, 2]:.6f}")
print(f"  B (defocus parameter y): {abcd[1, 3]:.6f}")

## 3. Collins FFT Diffraction Model

Implement the Collins integral (Fresnel diffraction) using the A and B parameters from the ABCD matrix.

The Collins integral propagates a field through an optical system characterized by ABCD matrix:

$$H(f_x, f_y) = \exp\left(-i\pi\lambda\frac{B}{A}(f_x^2 + f_y^2)\right)$$

where the effective defocus is $z_{\text{eff}} = B/A$.

In [ ]:
def collins_fft_propagation(input_field, input_size, A, B, wavelength):
    """Propagate a field using Collins integral (Fresnel diffraction).
    
    Parameters
    ----------
    input_field : jnp.ndarray, shape (N, N)
        Complex input field
    input_size : float
        Physical size of input grid (meters)
    A : float
        ABCD matrix A parameter (magnification)
    B : float
        ABCD matrix B parameter (related to defocus)
    wavelength : float
        Wavelength in meters
        
    Returns
    -------
    output_field : jnp.ndarray, shape (N, N)
        Propagated field at output plane
    """
    N = input_field.shape[0]
    dx = input_size / N
    
    # Frequency coordinates
    fx = jnp.fft.fftfreq(N, d=dx)
    fy = jnp.fft.fftfreq(N, d=dx)
    FX, FY = jnp.meshgrid(fx, fy)
    
    # Effective defocus from B/A
    z_eff = B / A if jnp.abs(A) > 1e-10 else 0.0
    
    # Collins transfer function (Fresnel kernel)
    H = jnp.exp(-1j * jnp.pi * wavelength * z_eff * (FX**2 + FY**2))
    
    # Apply FFT-based propagation
    U_input = jnp.fft.fft2(input_field)
    U_output = H * U_input
    output_field = jnp.fft.ifft2(U_output)
    
    # Add phase factor for propagation distance
    k = 2 * jnp.pi / wavelength
    output_field *= jnp.exp(1j * k * jnp.abs(z_eff))
    
    return output_field

print("Collins FFT propagation function defined")

## 4. Create Input Aperture

Create a circular aperture with appropriate padding.

In [ ]:
def create_circular_aperture(size, n_pixels, aperture_radius):
    """Create a circular aperture function.
    
    Parameters
    ----------
    size : float
        Physical size of the grid (meters)
    n_pixels : int
        Number of pixels per side
    aperture_radius : float
        Radius of the aperture (meters)
        
    Returns
    -------
    aperture : jnp.ndarray, shape (n_pixels, n_pixels)
        Binary aperture (1 inside, 0 outside)
    x, y : jnp.ndarray
        Coordinate arrays
    """
    # Create coordinate grid
    x = jnp.linspace(-size/2, size/2, n_pixels)
    y = jnp.linspace(-size/2, size/2, n_pixels)
    X, Y = jnp.meshgrid(x, y)
    
    # Create circular aperture
    R = jnp.sqrt(X**2 + Y**2)
    aperture = (R <= aperture_radius).astype(jnp.float32)
    
    return aperture, X, Y

# Create input aperture
input_aperture, X_in, Y_in = create_circular_aperture(
    INPUT_SIZE, INPUT_PIXELS, APERTURE_RADIUS
)

# Visualize input aperture
fig, ax = plt.subplots(1, 1, figsize=(6, 6))
extent = [-INPUT_SIZE/2*1e6, INPUT_SIZE/2*1e6, -INPUT_SIZE/2*1e6, INPUT_SIZE/2*1e6]
ax.imshow(input_aperture, extent=extent, origin='lower', cmap='gray')
ax.set_xlabel('x (μm)')
ax.set_ylabel('y (μm)')
ax.set_title('Input Aperture')
plt.tight_layout()
plt.show()

print(f"Input aperture created: {INPUT_PIXELS}×{INPUT_PIXELS} pixels")
print(f"Aperture diameter: {2*APERTURE_RADIUS*1e6:.2f} μm")
print(f"Grid size: {INPUT_SIZE*1e6:.2f} μm (×{INPUT_SIZE/(2*APERTURE_RADIUS):.1f} padding)")

## 5. Propagate Through System

Use Collins FFT to propagate the aperture through the optical system.

In [ ]:
# Create input field (uniform illumination through aperture)
input_field = input_aperture.astype(jnp.complex64)

# Get ABCD parameters
A_x = abcd[0, 0]
B_x = abcd[0, 2]
A_y = abcd[1, 1]
B_y = abcd[1, 3]

print(f"ABCD parameters:")
print(f"  A_x = {A_x:.6f}, B_x = {B_x:.6f}")
print(f"  A_y = {A_y:.6f}, B_y = {B_y:.6f}")
print(f"  z_eff_x = B_x/A_x = {B_x/A_x:.6f} m")
print(f"  z_eff_y = B_y/A_y = {B_y/A_y:.6f} m")

# Propagate using Collins FFT
# For simplicity, use x-direction parameters (assuming symmetric system)
output_field = collins_fft_propagation(
    input_field, INPUT_SIZE, A_x, B_x, WAVELENGTH
)

# Compute intensity
output_intensity = jnp.abs(output_field)**2

print(f"\nOutput field computed: {output_field.shape}")
print(f"Max intensity: {jnp.max(output_intensity):.6e}")

## 6. Zoom to Output Grid

Use `jax.scipy.ndimage` to zoom the solution onto the desired output grid.
We need to account for the magnification A and map to the correct physical scale.

In [ ]:
def zoom_to_output_grid(field, input_size, output_size, output_pixels, magnification):
    """Zoom and resample field to output grid.
    
    Parameters
    ----------
    field : jnp.ndarray
        Input field to resample
    input_size : float
        Physical size of input (meters)
    output_size : float
        Physical size of output (meters)
    output_pixels : int
        Number of pixels in output
    magnification : float
        System magnification (A parameter)
        
    Returns
    -------
    output_field : jnp.ndarray, shape (output_pixels, output_pixels)
        Resampled field
    """
    # Account for magnification
    # The field is magnified by A, so effective size is input_size * |A|
    magnified_size = input_size * jnp.abs(magnification)
    
    # Compute zoom factor
    # We want to map from magnified_size to output_size with output_pixels
    zoom_factor = (output_size / magnified_size) * (output_pixels / field.shape[0])
    
    print(f"  Input size: {input_size*1e6:.2f} μm")
    print(f"  Magnified size: {magnified_size*1e3:.2f} mm (M={magnification:.3f})")
    print(f"  Output size: {output_size*1e3:.2f} mm")
    print(f"  Zoom factor: {zoom_factor:.4f}")
    
    # Use jax.image.resize for zooming
    output_shape = (output_pixels, output_pixels)
    
    # For real fields
    if jnp.isrealobj(field):
        output_field = jax.image.resize(field, output_shape, method='bilinear')
    else:
        # For complex fields, resize real and imaginary parts separately
        real_part = jax.image.resize(jnp.real(field), output_shape, method='bilinear')
        imag_part = jax.image.resize(jnp.imag(field), output_shape, method='bilinear')
        output_field = real_part + 1j * imag_part
    
    return output_field

# Zoom output to desired grid
print("Zooming to output grid...")
output_field_zoomed = zoom_to_output_grid(
    output_field, INPUT_SIZE, OUTPUT_SIZE, OUTPUT_PIXELS, A_x
)
output_intensity_zoomed = jnp.abs(output_field_zoomed)**2

print(f"\nZoomed output: {output_intensity_zoomed.shape}")
print(f"Max intensity: {jnp.max(output_intensity_zoomed):.6e}")

## 7. Visualize Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot before zoom
extent_before = [-INPUT_SIZE/2*1e6, INPUT_SIZE/2*1e6, 
                  -INPUT_SIZE/2*1e6, INPUT_SIZE/2*1e6]
im0 = axes[0].imshow(output_intensity, extent=extent_before, 
                      origin='lower', cmap='hot')
axes[0].set_xlabel('x (μm)')
axes[0].set_ylabel('y (μm)')
axes[0].set_title('Diffracted Pattern (Input Scale)')
plt.colorbar(im0, ax=axes[0], label='Intensity')

# Plot after zoom
extent_after = [-OUTPUT_SIZE/2*1e3, OUTPUT_SIZE/2*1e3,
                -OUTPUT_SIZE/2*1e3, OUTPUT_SIZE/2*1e3]
im1 = axes[1].imshow(output_intensity_zoomed, extent=extent_after,
                      origin='lower', cmap='hot')
axes[1].set_xlabel('x (mm)')
axes[1].set_ylabel('y (mm)')
axes[1].set_title('Diffracted Pattern (Output Scale)')
plt.colorbar(im1, ax=axes[1], label='Intensity')

plt.tight_layout()
plt.show()

# Plot cross-sections
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Horizontal cross-section
mid = OUTPUT_PIXELS // 2
x_axis = jnp.linspace(-OUTPUT_SIZE/2, OUTPUT_SIZE/2, OUTPUT_PIXELS) * 1e3
axes[0].plot(x_axis, output_intensity_zoomed[mid, :])
axes[0].set_xlabel('x (mm)')
axes[0].set_ylabel('Intensity')
axes[0].set_title('Horizontal Cross-Section')
axes[0].grid(True, alpha=0.3)

# Vertical cross-section
axes[1].plot(x_axis, output_intensity_zoomed[:, mid])
axes[1].set_xlabel('y (mm)')
axes[1].set_ylabel('Intensity')
axes[1].set_title('Vertical Cross-Section')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Define Loss Function

Create a loss function that compares the diffraction model with a target.
For now, we'll use this as a template for optimization.

In [ ]:
def forward_model_loss(params, target_intensity=None):
    """Compute loss for the forward model.
    
    Parameters
    ----------
    params : dict
        Dictionary with keys: 'z1', 'z2', 'z3', 'f1', 'f2'
    target_intensity : jnp.ndarray, optional
        Target intensity pattern to match
        
    Returns
    -------
    loss : float
        Mean squared error or other metric
    predicted_intensity : jnp.ndarray
        Predicted intensity pattern
    """
    z1, z2, z3 = params['z1'], params['z2'], params['z3']
    f1, f2 = params['f1'], params['f2']
    
    # Get ABCD matrix
    abcd = get_abcd_matrix(z1, z2, z3, f1, f2)
    A_x, B_x = abcd[0, 0], abcd[0, 2]
    
    # Create input field
    input_aperture, _, _ = create_circular_aperture(
        INPUT_SIZE, INPUT_PIXELS, APERTURE_RADIUS
    )
    input_field = input_aperture.astype(jnp.complex64)
    
    # Propagate
    output_field = collins_fft_propagation(
        input_field, INPUT_SIZE, A_x, B_x, WAVELENGTH
    )
    
    # Zoom to output grid
    output_field_zoomed = zoom_to_output_grid(
        output_field, INPUT_SIZE, OUTPUT_SIZE, OUTPUT_PIXELS, A_x
    )
    predicted_intensity = jnp.abs(output_field_zoomed)**2
    
    # Compute loss
    if target_intensity is not None:
        loss = jnp.mean((predicted_intensity - target_intensity)**2)
    else:
        # If no target, return negative of some quality metric
        # For example, could maximize central peak intensity
        loss = -jnp.max(predicted_intensity)
    
    return loss, predicted_intensity

# Test the loss function
test_params = {'z1': Z1, 'z2': Z2, 'z3': Z3, 'f1': F1, 'f2': F2}
loss, pred_intensity = forward_model_loss(test_params)
print(f"Test loss: {loss:.6e}")
print(f"Predicted intensity shape: {pred_intensity.shape}")

## 9. Bayesian Optimization with Optuna

Set up Bayesian optimization using Optuna to optimize lens parameters.
This is a template - in practice you would have target data to fit.

In [ ]:
try:
    import optuna
    optuna_available = True
except ImportError:
    print("Optuna not installed. Install with: pip install optuna")
    optuna_available = False

if optuna_available:
    # For demonstration, create synthetic target data
    # In practice, this would be your experimental data
    target_params = {'z1': 0.0, 'z2': 0.12, 'z3': 0.55, 'f1': 0.055, 'f2': 0.16}
    _, target_intensity = forward_model_loss(target_params)
    
    # Add some noise to make it realistic
    np.random.seed(42)
    target_intensity = target_intensity + 0.01 * jnp.std(target_intensity) * np.random.randn(*target_intensity.shape)
    target_intensity = jnp.maximum(target_intensity, 0)  # Ensure non-negative
    
    print("Synthetic target created")
    print(f"Target parameters: {target_params}")
    
    # Define objective function for Optuna
    def objective(trial):
        """Optuna objective function."""
        # Suggest parameters within reasonable bounds
        params = {
            'z1': 0.0,  # Fixed at input plane
            'z2': trial.suggest_float('z2', 0.05, 0.2),
            'z3': trial.suggest_float('z3', 0.3, 0.8),
            'f1': trial.suggest_float('f1', 0.02, 0.1),
            'f2': trial.suggest_float('f2', 0.1, 0.25),
        }
        
        # Compute loss
        try:
            loss, _ = forward_model_loss(params, target_intensity)
            return float(loss)
        except Exception as e:
            print(f"Error in trial: {e}")
            return float('inf')
    
    print("\nOptuna objective function defined")
    print("Ready to run optimization with: study.optimize(objective, n_trials=N)")

### Run Optimization (Example)

Run a small number of trials as a demonstration.

In [ ]:
if optuna_available:
    # Create study
    study = optuna.create_study(direction='minimize', study_name='two_lens_optimization')
    
    # Run optimization (small number of trials for demo)
    n_trials = 10  # Increase for real optimization (e.g., 100-200)
    print(f"Running {n_trials} optimization trials...")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    
    # Show results
    print("\n" + "="*60)
    print("OPTIMIZATION RESULTS")
    print("="*60)
    print(f"\nBest loss: {study.best_value:.6e}")
    print(f"\nBest parameters:")
    for key, value in study.best_params.items():
        print(f"  {key}: {value:.6f}")
    
    print(f"\nTrue parameters (target):")
    for key, value in target_params.items():
        if key in study.best_params:
            print(f"  {key}: {value:.6f}")
    
    print(f"\nNumber of trials completed: {len(study.trials)}")
    
    # Visualize optimization history
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    trials = study.trials
    values = [t.value for t in trials if t.value is not None and t.value != float('inf')]
    ax.plot(values, 'o-')
    ax.set_xlabel('Trial')
    ax.set_ylabel('Loss')
    ax.set_title('Optimization History')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Compare best fit with target
    best_params_full = {'z1': 0.0, **study.best_params}
    _, best_intensity = forward_model_loss(best_params_full, target_intensity)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    extent = [-OUTPUT_SIZE/2*1e3, OUTPUT_SIZE/2*1e3,
              -OUTPUT_SIZE/2*1e3, OUTPUT_SIZE/2*1e3]
    
    im0 = axes[0].imshow(target_intensity, extent=extent, origin='lower', cmap='hot')
    axes[0].set_title('Target')
    axes[0].set_xlabel('x (mm)')
    axes[0].set_ylabel('y (mm)')
    plt.colorbar(im0, ax=axes[0])
    
    im1 = axes[1].imshow(best_intensity, extent=extent, origin='lower', cmap='hot')
    axes[1].set_title('Best Fit')
    axes[1].set_xlabel('x (mm)')
    axes[1].set_ylabel('y (mm)')
    plt.colorbar(im1, ax=axes[1])
    
    residual = jnp.abs(target_intensity - best_intensity)
    im2 = axes[2].imshow(residual, extent=extent, origin='lower', cmap='hot')
    axes[2].set_title('Absolute Residual')
    axes[2].set_xlabel('x (mm)')
    axes[2].set_ylabel('y (mm)')
    plt.colorbar(im2, ax=axes[2])
    
    plt.tight_layout()
    plt.show()
    
else:
    print("Skipping optimization - Optuna not available")

## Summary

This notebook demonstrates:

1. **Forward Model**: Built a two-lens optical system using temgym_core components
2. **ABCD Matrix**: Computed transfer matrix via JAX automatic differentiation
3. **Collins FFT**: Implemented Fresnel diffraction using ABCD parameters
4. **Zooming**: Resampled output to desired detector grid
5. **Loss Function**: Defined comparison metric for optimization
6. **Bayesian Optimization**: Set up Optuna for parameter optimization

### Next Steps:

- Increase number of trials for more thorough optimization
- Add constraints based on physical limits
- Include priors from manufacturer specifications
- Extend to multiple defocus planes
- Add lens wobble for better parameter identifiability
- Compare with full ray-traced simulation